In [ ]:
# Installing requirements
!pip install langchain langchain-community langchain-groq sentence-transformers  faiss-cpu pypdf

In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

In [4]:
# Using groq's API
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
groq_api_key = os.environ["GROQ_API_KEY"]

In [ ]:
# Testing the model
llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

response = llm.invoke("Explain RAG in one paragraph")

print(response.content)

In [ ]:
# Uploading your file which you want to use in RAG
uploaded = files.upload()

In [7]:
loader = PyPDFLoader("Chapter 3 - Deadlock.pdf")

documents = loader.load()

In [ ]:
#Each PDF page becomes a Document object.
print(documents[14])

In [ ]:
print(documents[0].page_content)

In [10]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [11]:
chunks = text_splitter.split_documents(documents)

In [ ]:
len(chunks)

In [ ]:
print(chunks[0].page_content)

In [ ]:
print(chunks[1].metadata)

In [ ]:
# Embedding
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [16]:
vector = embeddings.embed_query("What is deadlock?")

In [ ]:
len(vector)

In [18]:
# Storing Embedding in vectore database
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [19]:
# Testing Retrieval from vetore database
query = "What are the conditions for deadlock?"

results = vectorstore.similarity_search(query)

In [ ]:
print(results[0].page_content)

In [ ]:
query = "Example of Banker’s Algorithm:"

results = vectorstore.similarity_search(query)

print(results[0].page_content)

In [22]:
retriever = vectorstore.as_retriever()

In [ ]:
retriever.invoke("What are the conditions for deadlock?")

In [24]:
# System prompt
from langchain_core.prompts import PromptTemplate
prompt_template = """
You are a helpful assistant.

Answer the question ONLY from the provided context.

If the answer is not in the context, say:
"I could not find the answer in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [25]:
# main RAG's function for running
def ask_rag(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(final_prompt)

    return response.content

In [ ]:
answer = ask_rag("explain Banker’s Algorithm")

print(answer)

In [ ]:
# as we can see, RAG assistant doesn't answer non relevant questions!
ask_rag("Who won the FIFA World Cup 2022?")